# Oxylipin pretreatment
Normalize dataset by division /D60 values

In [1]:
import pandas as pd
import numpy as np

In [2]:
# Charger le fichier Excel
file_path = "/Users/loictalignani/research/project/lipidomics/data/raw_data/oxylipines.xlsx"
xls = pd.ExcelFile(file_path)

# Charger les données des onglets
species = pd.read_excel(xls, sheet_name="data")
patient_data = pd.read_excel(xls, sheet_name="order")

In [3]:
species

,feature,KT-312D10,BS-364D0,KT-412D60,KT-974D03,JV-138D0,BS-671D03,KT-723D10,KT-705D60,KT-434D03,...,KT-537D0,KT-313D03,JV-035D10,BS-377D03,KT-880D60,KT-434D0,JV-035D03,KT-412D0,BS-671D10,KT-974D60
0,9-HODE,14.865283,21.497567,10.846900,89.558500,35.857650,27.483217,14.128983,7.418400,28.676300,...,22.163550,10.877283,23.700650,39.977683,30.411150,13.210950,20.561650,13.653850,13.947750,17.897417
1,13-HODE,27.323333,47.252867,13.739850,108.001150,46.803067,33.524733,47.747450,19.417933,87.156267,...,33.607200,20.156767,47.817767,39.353600,63.388833,67.179217,82.700817,66.622750,89.898633,29.086750
2,"9,10-DiHOME",1.970350,2.820100,4.621233,13.864950,2.510167,0.598733,2.749933,1.201033,2.399867,...,5.272700,1.531750,3.214550,0.706083,4.146717,11.880800,4.403850,7.940333,7.006133,1.669117
3,"12,13-DiHOME",2.482433,2.256850,4.976750,19.789067,2.185733,0.613150,3.633650,0.937550,2.547783,...,9.651200,1.415483,3.704800,0.773417,4.887717,14.942933,4.206400,3.897400,6.528050,2.863050
4,9-OxoODE,1.736617,0.718367,0.000000,6.359050,0.563050,0.182233,0.134317,0.136050,0.545383,...,0.309400,0.456867,0.318400,0.863067,0.718200,0.000000,0.765183,0.825683,0.135683,0.185667
5,13-OxoODE,2.087400,0.942467,0.000000,7.480950,2.314950,0.626550,0.000000,0.234333,0.696083,...,0.000000,0.000000,1.037783,5.671217,1.413917,0.245317,0.687183,1.131417,0.437083,0.000000
6,"9,10,13-trihydroxy-11-octadecenoic acid",0.365767,0.308683,0.470983,5.466883,0.000000,0.198700,0.146267,0.245300,0.929750,...,1.623800,0.653533,0.445483,0.000000,0.856550,2.213133,0.252400,0.224117,0.355400,0.000000
7,"9S,12S,13S-trihydroxy-10E-octadecenoic acid",1.444417,1.234267,2.728650,15.297733,0.488017,0.925133,1.894683,1.061883,2.826267,...,7.591317,2.235617,1.810333,0.399067,3.932633,7.085850,1.777633,1.343283,5.634533,1.680733
8,"8,9-EET",0.762267,2.258633,0.000000,2.601133,0.828633,1.540600,2.979100,1.694300,2.106700,...,2.183283,0.411917,0.421850,1.616133,4.455650,10.225200,0.441017,1.487100,0.422867,0.000000
9,5-HETE,96.875467,118.989883,125.045100,156.136400,173.930333,193.978650,93.744783,49.465500,45.070200,...,204.277067,41.275050,87.794900,250.758000,50.098317,34.865733,5.566117,11.948700,97.012750,28.531367


In [4]:
species.dtypes

feature       object
KT-312D10    float64
BS-364D0     float64
KT-412D60    float64
KT-974D03    float64
              ...   
KT-434D0     float64
JV-035D03    float64
KT-412D0     float64
BS-671D10    float64
KT-974D60    float64
Length: 160, dtype: object

In [5]:
# Extraire les noms de colonnes correspondant aux échantillons
sample_columns = [
    col for col in species.columns if "D0" in col or "D03" in col or "D10" in col or "D60" in col]

In [6]:
# Identifier les patients en récupérant les préfixes des colonnes
patients = set(col.split('D')[0]
               for col in species.columns[1:])  # Exclure 'feature'
patients

{'BS-064',
 'BS-082',
 'BS-336',
 'BS-346',
 'BS-351',
 'BS-364',
 'BS-377',
 'BS-671',
 'JV-035',
 'JV-138',
 'JV-148',
 'JV-157',
 'KT-193',
 'KT-247',
 'KT-312',
 'KT-313',
 'KT-347',
 'KT-352',
 'KT-412',
 'KT-417',
 'KT-434',
 'KT-445',
 'KT-515',
 'KT-522',
 'KT-525',
 'KT-537',
 'KT-538',
 'KT-539',
 'KT-612',
 'KT-663',
 'KT-695',
 'KT-705',
 'KT-716',
 'KT-718',
 'KT-723',
 'KT-771',
 'KT-805',
 'KT-880',
 'KT-926',
 'KT-927',
 'KT-928',
 'KT-974'}

In [7]:
offset = 1  # Offset à ajouter pour éviter la division par zéro

In [8]:
# Appliquer l'offset à toutes les colonnes de patients
for patient in patients:
    patient_cols = [col for col in species.columns if col.startswith(patient)]
    # Ajout de l'offset pour toutes les colonnes du patient
    species[patient_cols] += offset

In [9]:
# Calculer les ratios et créer les nouvelles colonnes
for patient in patients:
    d0_col, d3_col, d10_col, d60_col = f"{patient}D0", f"{patient}D03", f"{patient}D10", f"{patient}D60"

    if d60_col in species.columns:
        species[f"{patient}_D0_D60"] = species[d0_col] / species[d60_col]
        species[f"{patient}_D3_D60"] = species[d3_col] / species[d60_col]
        species[f"{patient}_D10_D60"] = species[d10_col] / species[d60_col]

/var/folders/d3/l5snwfsj7yb7zrln2fvzwpqr0000gn/T/ipykernel_75006/1705873891.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  species[f"{patient}_D0_D60"] = species[d0_col] / species[d60_col]
/var/folders/d3/l5snwfsj7yb7zrln2fvzwpqr0000gn/T/ipykernel_75006/1705873891.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  species[f"{patient}_D3_D60"] = species[d3_col] / species[d60_col]
/var/folders/d3/l5snwfsj7yb7zrln2fvzwpqr0000gn/T/ipykernel_75006/1705873891.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usu

In [10]:
species

,feature,KT-312D10,BS-364D0,KT-412D60,KT-974D03,JV-138D0,BS-671D03,KT-723D10,KT-705D60,KT-434D03,...,KT-723_D10_D60,BS-082_D0_D60,BS-082_D3_D60,BS-082_D10_D60,JV-138_D0_D60,JV-138_D3_D60,JV-138_D10_D60,BS-364_D0_D60,BS-364_D3_D60,BS-364_D10_D60
0,9-HODE,15.865283,22.497567,11.846900,90.558500,36.857650,28.483217,15.128983,8.418400,29.676300,...,0.632885,12.144158,20.901049,1.121032,0.312892,0.142642,0.237797,0.113751,0.355322,0.196746
1,13-HODE,28.323333,48.252867,14.739850,109.001150,47.803067,34.524733,48.747450,20.417933,88.156267,...,1.163771,13.251886,21.811315,4.018942,0.306218,0.194506,0.333465,0.190865,0.241290,0.220880
2,"9,10-DiHOME",2.970350,3.820100,5.621233,14.864950,3.510167,1.598733,3.749933,2.201033,3.399867,...,1.152702,0.767151,0.711355,7.939127,0.930055,2.618850,0.975774,0.442874,1.466490,0.260309
3,"12,13-DiHOME",3.482433,3.256850,5.976750,20.789067,3.185733,1.613150,4.633650,1.937550,3.547783,...,1.533170,0.349541,0.325722,3.176758,0.874305,1.885507,0.810940,0.521077,4.476540,0.270059
4,9-OxoODE,2.736617,1.718367,1.000000,7.359050,1.563050,1.182233,1.134317,1.136050,1.545383,...,1.004961,1.899190,2.671447,1.587983,0.968663,0.861233,0.986604,1.204181,2.234139,0.752371
5,13-OxoODE,3.087400,1.942467,1.000000,8.480950,3.314950,1.626550,1.000000,1.234333,1.696083,...,1.000000,7.438405,6.921456,1.803543,3.314950,1.421633,2.054017,0.450507,0.478230,0.337401
6,"9,10,13-trihydroxy-11-octadecenoic acid",1.365767,1.308683,1.470983,6.466883,1.000000,1.198700,1.146267,1.245300,1.929750,...,0.925715,0.983177,2.163615,4.668123,0.745221,1.044465,1.014631,0.833317,0.822715,0.740605
7,"9S,12S,13S-trihydroxy-10E-octadecenoic acid",2.444417,2.234267,3.728650,16.297733,1.488017,1.925133,2.894683,2.061883,3.826267,...,1.274536,0.545924,1.278083,5.427426,0.671953,1.189248,1.224132,0.719104,1.416847,0.669528
8,"8,9-EET",1.762267,3.258633,1.000000,3.601133,1.828633,2.540600,3.979100,2.694300,3.106700,...,1.323396,0.652186,0.929987,0.525833,1.828633,1.418333,2.751383,1.353084,0.758004,0.663289
9,5-HETE,97.875467,119.989883,126.045100,157.136400,174.930333,194.978650,94.744783,50.465500,46.070200,...,0.560043,12.018189,15.827305,1.522597,0.685547,0.431333,0.526913,0.151542,0.027557,0.333347


In [13]:
family_col = [
    col for col in species.columns if "feature" in col]
family_col

['feature']

In [16]:
ratio_cols_D0 = [
    col for col in species.columns if "D0_D60" in col]
ratio_cols_D3 = [
    col for col in species.columns if "D3_D60" in col]
ratio_cols_D10 = [
    col for col in species.columns if "D10_D60" in col]

In [17]:
ratio_data_D0 = species[family_col + ratio_cols_D0]
ratio_data_D3 = species[family_col + ratio_cols_D3]
ratio_data_D10 = species[family_col + ratio_cols_D10]

In [24]:
ordered_columns = [
    "feature",
    "KT-539_D0_D60", "BS-671_D0_D60", "KT-805_D0_D60", "BS-336_D0_D60",
    "BS-082_D0_D60", "BS-351_D0_D60", "BS-377_D0_D60", "KT-313_D0_D60",
    "BS-364_D0_D60", "KT-880_D0_D60", "JV-138_D0_D60", "KT-412_D0_D60",
    "KT-771_D0_D60", "KT-723_D0_D60", "KT-538_D0_D60", "KT-716_D0_D60",
    "KT-974_D0_D60", "KT-537_D0_D60", "KT-445_D0_D60", "BS-346_D0_D60",
    "KT-663_D0_D60", "KT-695_D0_D60", "BS-064_D0_D60", "KT-705_D0_D60",
    "KT-347_D0_D60", "KT-515_D0_D60", "KT-718_D0_D60", "KT-417_D0_D60",
    "KT-522_D0_D60", "KT-312_D0_D60", "JV-157_D0_D60", "KT-193_D0_D60",
    "JV-035_D0_D60", "KT-434_D0_D60", "KT-247_D0_D60", "KT-612_D0_D60",
    "KT-525_D0_D60", "KT-352_D0_D60", "JV-148_D0_D60"
]

ratio_data_D0 = ratio_data_D0[ordered_columns]

In [25]:
ordered_columns = [
    "feature",
    "KT-539_D3_D60", "BS-671_D3_D60", "KT-805_D3_D60", "BS-336_D3_D60",
    "BS-082_D3_D60", "BS-351_D3_D60", "BS-377_D3_D60", "KT-313_D3_D60",
    "BS-364_D3_D60", "KT-880_D3_D60", "JV-138_D3_D60", "KT-412_D3_D60",
    "KT-771_D3_D60", "KT-723_D3_D60", "KT-538_D3_D60", "KT-716_D3_D60",
    "KT-974_D3_D60", "KT-537_D3_D60", "KT-445_D3_D60", "BS-346_D3_D60",
    "KT-663_D3_D60", "KT-695_D3_D60", "BS-064_D3_D60", "KT-705_D3_D60",
    "KT-347_D3_D60", "KT-515_D3_D60", "KT-718_D3_D60", "KT-417_D3_D60",
    "KT-522_D3_D60", "KT-312_D3_D60", "JV-157_D3_D60", "KT-193_D3_D60",
    "JV-035_D3_D60", "KT-434_D3_D60", "KT-247_D3_D60", "KT-612_D3_D60",
    "KT-525_D3_D60", "KT-352_D3_D60", "JV-148_D3_D60"
]

ratio_data_D3 = ratio_data_D3[ordered_columns]

In [26]:
ordered_columns = [
    "feature",
    "KT-539_D10_D60", "BS-671_D10_D60", "KT-805_D10_D60", "BS-336_D10_D60",
    "BS-082_D10_D60", "BS-351_D10_D60", "BS-377_D10_D60", "KT-313_D10_D60",
    "BS-364_D10_D60", "KT-880_D10_D60", "JV-138_D10_D60", "KT-412_D10_D60",
    "KT-771_D10_D60", "KT-723_D10_D60", "KT-538_D10_D60", "KT-716_D10_D60",
    "KT-974_D10_D60", "KT-537_D10_D60", "KT-445_D10_D60", "BS-346_D10_D60",
    "KT-663_D10_D60", "KT-695_D10_D60", "BS-064_D10_D60", "KT-705_D10_D60",
    "KT-347_D10_D60", "KT-515_D10_D60", "KT-718_D10_D60", "KT-417_D10_D60",
    "KT-522_D10_D60", "KT-312_D10_D60", "JV-157_D10_D60", "KT-193_D10_D60",
    "JV-035_D10_D60", "KT-434_D10_D60", "KT-247_D10_D60", "KT-612_D10_D60",
    "KT-525_D10_D60", "KT-352_D10_D60", "JV-148_D10_D60"
]

ratio_data_D10 = ratio_data_D10[ordered_columns]

In [28]:
# Sauvegarder le nouveau fichier
outfileD0 = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/Oxylipines/Ratio_lipid_abundance_oxylipins_D0.tsv"
ratio_data_D0.to_csv(outfileD0, sep="\t", index=False)

print(f"Fichier enregistré : {outfileD0}")

# Sauvegarder le nouveau fichier
outfileD03 = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/Oxylipines/Ratio_lipid_abundance_oxylipins_D03.tsv"
ratio_data_D3.to_csv(outfileD03, sep="\t", index=False)

print(f"Fichier enregistré : {outfileD03}")

# Sauvegarder le nouveau fichier
outfileD10 = "/Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/Oxylipines/Ratio_lipid_abundance_oxylipins_D10.tsv"
ratio_data_D10.to_csv(outfileD10, sep="\t", index=False)

print(f"Fichier enregistré : {outfileD10}")

Fichier enregistré : /Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/Oxylipines/Ratio_lipid_abundance_oxylipins_D0.tsv
Fichier enregistré : /Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/Oxylipines/Ratio_lipid_abundance_oxylipins_D03.tsv
Fichier enregistré : /Users/loictalignani/research/project/lipidomics/data/lipidsig_datasets/Oxylipines/Ratio_lipid_abundance_oxylipins_D10.tsv
